<a href="https://colab.research.google.com/github/hollymunck-glitch/is_4487_base/blob/main/Assignments/assignment_11_regression.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# IS 4487 Assignment 11: Predicting Airbnb Prices with Regression

In this assignment, you will:
- Load the Airbnb dataset you cleaned and transformed in Assignment 7
- Build a linear regression model to predict listing price
- Interpret which features most affect price
- Try to improve your model using only the most impactful predictors
- Practice explaining your findings to a business audience like a host, pricing strategist, or city partner

## Why This Matters

Pricing is one of the most important levers for hosts and Airbnb’s business teams. Understanding what drives price — and being able to predict it accurately — helps improve search results, revenue management, and guest satisfaction.

This assignment gives you hands-on practice turning a cleaned dataset into a predictive model. You’ll focus not just on code, but on what the results mean and how you’d communicate them to stakeholders.

<a href="https://colab.research.google.com/github/Stan-Pugsley/is_4487_base/blob/main/Assignments/assignment_11_regression.ipynb" target="_parent">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>



## Original Source: Dataset Description

The dataset you'll be using is a **detailed Airbnb listing file**, available from [Inside Airbnb](https://insideairbnb.com/get-the-data/).

Each row represents one property listing. The columns include:

- **Host attributes** (e.g., host ID, host name, host response time)
- **Listing details** (e.g., price, room type, minimum nights, availability)
- **Location data** (e.g., neighborhood, latitude/longitude)
- **Property characteristics** (e.g., number of bedrooms, amenities, accommodates)
- **Calendar/booking variables** (e.g., last review date, number of reviews)

The schema is consistent across cities, so you can expect similar columns regardless of the location you choose.

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score


## 1. Load Your Transformed Airbnb Dataset

**Business framing:**  
Before building any models, we must start with clean, prepared data. In Assignment 7, you exported a cleaned version of your Airbnb dataset. You’ll now import that file for analysis.

### Do the following:
- Import your CSV file called `cleaned_airbnb_data_7.csv`.   (Note: If you had significant errors with assignment 7, you can use the file named "airbnb_listings.csv" in the DataSets folder on GitHub as a backup starting point.)
- Use `pandas` to load and preview the dataset

### In Your Response:
1. What does the dataset include?
2. How many rows and columns are present?


In [2]:
df = pd.read_csv('/content/airbnb_listings.csv')
display(df.head())

,id,listing_url,scrape_id,last_scraped,source,name,description,neighborhood_overview,host_url,host_name,...,review_scores_communication,review_scores_location,review_scores_value,license,instant_bookable,calculated_host_listings_count,calculated_host_listings_count_entire_homes,calculated_host_listings_count_private_rooms,calculated_host_listings_count_shared_rooms,reviews_per_month
0,2992450,https://www.airbnb.com/rooms/2992450,20250804133828,2025-08-04,city scrape,Luxury 2 bedroom apartment,The apartment is located in a quiet neighborho...,NaN,https://www.airbnb.com/users/show/4621559,Kenneth,...,4.56,3.22,3.67,NaN,0,1,1,0,0,0.07
1,3820211,https://www.airbnb.com/rooms/3820211,20250804133828,2025-08-04,city scrape,Funky Urban Gem: Prime Central Location - Park...,Step into the charming and comfy 1BR/1BA apart...,Overview<br /><br />The lovely apartment is lo...,https://www.airbnb.com/users/show/19648678,Terra,...,4.81,4.81,4.77,NaN,0,4,4,0,0,2.32
2,5651579,https://www.airbnb.com/rooms/5651579,20250804133828,2025-08-04,city scrape,Large studio apt by Capital Center & ESP@,"Spacious studio with hardwood floors, fully eq...",The neighborhood is very eclectic. We have a v...,https://www.airbnb.com/users/show/29288920,Gregg,...,4.88,4.76,4.64,NaN,0,2,1,1,0,2.97
3,6623339,https://www.airbnb.com/rooms/6623339,20250804133828,2025-08-04,city scrape,Bright & Cozy City Stay · Top Location + Parking!,Step into the charming and comfy 1BR/1BA apart...,Overview<br /><br />The lovely apartment is lo...,https://www.airbnb.com/users/show/19648678,Terra,...,4.70,4.80,4.72,NaN,0,4,4,0,0,2.68
4,9005989,https://www.airbnb.com/rooms/9005989,20250804133828,2025-08-04,city scrape,"Studio in The heart of Center SQ, in Albany NY",(21 years of age or older ONLY) NON- SMOKING.....,"There are many shops, restaurants, bars, museu...",https://www.airbnb.com/users/show/17766924,Sugey,...,4.93,4.87,4.77,NaN,0,1,1,0,0,5.67


### ✍️ Your Response: 🔧
1. The dataset includes many details about Airbnb listings such as host information, listing details, location data, and property characteristics.

2. Based on the output from the previous cell, there are 459 rows and 77 columns in the dataset.


## 2. Drop Columns Not Useful for Modeling

**Business framing:**  
Some columns — like post IDs or text — may not help us predict price and could add noise or bias.

### Do the following:
- Drop columns like `post_id`, `title`, `descr`, `details`, and `address` if they’re still in your dataset

### In Your Response:
1. What columns did you drop, and why?
2. What risks might occur if you included them in your model?


In [5]:
columns_to_drop = ['id', 'listing_url', 'scrape_id', 'last_scraped', 'source', 'name', 'description', 'neighborhood_overview', 'picture_url', 'host_url', 'host_about', 'host_thumbnail_url', 'host_picture_url', 'host_neighbourhood', 'host_verifications', 'calendar_last_scraped', 'first_review', 'last_review', 'license', 'neighbourhood_group_cleansed', 'bathrooms', 'calendar_updated']
columns_to_drop_existing = [col for col in columns_to_drop if col in df.columns]
df_cleaned = df.drop(columns=columns_to_drop_existing)
display(df_cleaned.head())

,host_name,host_since,host_location,host_response_time,host_response_rate,host_acceptance_rate,host_is_superhost,host_listings_count,host_total_listings_count,host_has_profile_pic,...,review_scores_checkin,review_scores_communication,review_scores_location,review_scores_value,instant_bookable,calculated_host_listings_count,calculated_host_listings_count_entire_homes,calculated_host_listings_count_private_rooms,calculated_host_listings_count_shared_rooms,reviews_per_month
0,Kenneth,2013-01-07,"New York, NY",NaN,NaN,50%,f,1,5,t,...,4.22,4.56,3.22,3.67,0,1,1,0,0,0.07
1,Terra,2014-08-07,"Albany, NY",within an hour,100%,100%,t,4,6,t,...,4.85,4.81,4.81,4.77,0,4,4,0,0,2.32
2,Gregg,2015-03-13,"Albany, NY",within an hour,100%,99%,f,2,2,t,...,4.81,4.88,4.76,4.64,0,2,1,1,0,2.97
3,Terra,2014-08-07,"Albany, NY",within an hour,100%,100%,t,4,6,t,...,4.83,4.70,4.80,4.72,0,4,4,0,0,2.68
4,Sugey,2014-07-07,"Albany, NY",NaN,NaN,100%,t,1,1,t,...,4.95,4.93,4.87,4.77,0,1,1,0,0,5.67


### ✍️ Your Response: 🔧
1. I dropped columns such as id, listing_url, scrape_id, last_scraped, source, name, description, neighborhood_overview, picture_url, host_url, host_about, host_thumbnail_url, host_picture_url, host_neighbourhood, host_verifications, calendar_last_scraped, first_review, last_review, license, neighbourhood_group_cleansed, bathrooms, and calendar_updated. These columns contain identifiers, text descriptions, URLs, or redundant information that are not directly useful for predicting price in a linear regression model.

2. Including these columns in the model could introduce noise and bias. Text-based columns are difficult for a linear model to interpret directly, and identifier columns are unique to each listing and don't provide generalizable information. Including them could lead to overfitting, where the model performs well on the training data but poorly on new, unseen data.

## 3. Explore Relationships Between Numeric Features

**Business framing:**  
Understanding how features relate to each other — and to the target — helps guide feature selection and modeling.

### Do the following:
- Generate a correlation matrix
- Identify which variables are strongly related to `price`

### In Your Response:
1. Which variables had the strongest positive or negative correlation with price?
2. Which variables might be useful predictors?


In [6]:
correlation_matrix = df_cleaned.corr(numeric_only=True)
display(correlation_matrix)

,host_listings_count,host_total_listings_count,latitude,longitude,accommodates,bedrooms,beds,price,minimum_nights,maximum_nights,...,review_scores_checkin,review_scores_communication,review_scores_location,review_scores_value,instant_bookable,calculated_host_listings_count,calculated_host_listings_count_entire_homes,calculated_host_listings_count_private_rooms,calculated_host_listings_count_shared_rooms,reviews_per_month
host_listings_count,1.000000,0.973960,-0.070382,0.111195,0.037236,0.041395,0.057712,-0.014677,0.034032,0.166138,...,-0.248577,-0.341142,-0.241995,-0.375941,0.277760,0.015981,0.057724,-0.061563,NaN,-0.116031
host_total_listings_count,0.973960,1.000000,-0.072907,0.099717,0.057426,0.061755,0.086508,-0.006580,0.005437,0.178693,...,-0.205193,-0.302003,-0.212705,-0.304869,0.252688,0.007070,0.048405,-0.062113,NaN,-0.103788
latitude,-0.070382,-0.072907,1.000000,-0.548236,0.006482,0.016919,0.021012,-0.014666,0.146688,-0.099903,...,0.036571,0.101162,0.108243,0.066316,0.091104,0.120997,-0.017696,0.227452,NaN,-0.119777
longitude,0.111195,0.099717,-0.548236,1.000000,-0.083466,-0.215899,-0.138303,-0.118913,-0.120240,-0.006670,...,-0.185848,-0.166284,-0.332212,-0.214617,0.043940,-0.047019,0.109600,-0.245058,NaN,0.045170
accommodates,0.037236,0.057426,0.006482,-0.083466,1.000000,0.809299,0.828875,0.579588,-0.117221,0.056143,...,0.011304,0.018571,-0.005875,0.009148,0.014482,-0.025960,0.100762,-0.196689,NaN,0.075746
bedrooms,0.041395,0.061755,0.016919,-0.215899,0.809299,1.000000,0.784348,0.499286,-0.073682,0.018216,...,-0.029907,-0.005564,-0.007703,0.016780,-0.033193,-0.040207,0.018162,-0.094575,NaN,-0.049262
beds,0.057712,0.086508,0.021012,-0.138303,0.828875,0.784348,1.000000,0.547032,-0.132096,0.060530,...,-0.026797,0.001802,-0.042161,0.000307,0.014030,-0.135324,-0.057550,-0.140551,NaN,0.078050
price,-0.014677,-0.006580,-0.014666,-0.118913,0.579588,0.499286,0.547032,1.000000,-0.075122,0.017992,...,-0.103223,-0.131798,-0.015814,0.018269,-0.013742,0.015773,0.033206,-0.024513,NaN,-0.090843
minimum_nights,0.034032,0.005437,0.146688,-0.120240,-0.117221,-0.073682,-0.132096,-0.075122,1.000000,-0.008418,...,-0.083556,0.005171,-0.091679,-0.071429,-0.037757,-0.067989,-0.121811,0.073141,NaN,-0.265131
maximum_nights,0.166138,0.178693,-0.099903,-0.006670,0.056143,0.018216,0.060530,0.017992,-0.008418,1.000000,...,0.038206,-0.016282,0.042969,-0.010976,0.015429,-0.025223,0.093624,-0.184583,NaN,-0.039948


### ✍️ Your Response: 🔧
1. Based on the correlation matrix, the variables with the strongest positive or negative correlation with price are:

accommodates (positive correlation)
bedrooms (positive correlation)
beds (positive correlation)
estimated_revenue_l365d (positive correlation)
review_scores_value (negative correlation)
review_scores_communication (negative correlation)
review_scores_rating (negative correlation)
review_scores_cleanliness (negative correlation)
review_scores_accuracy (negative correlation)
review_scores_checkin (negative correlation)
review_scores_location (negative correlation)
reviews_per_month (negative correlation)

2. Variables that might be useful predictors are those with a relatively higher absolute correlation with price. These include accommodates, bedrooms, beds, estimated_revenue_l365d, and the review_scores variables. While some review scores have negative correlations, this doesn't necessarily mean they are not useful predictors; it just indicates an inverse relationship with price in this dataset

## 4. Define Features and Target Variable

**Business framing:**  
To build a regression model, you need to define what you’re predicting (target) and what you’re using to make that prediction (features).

### Do the following:
- Set `price` as your target variable
- Remove `price` from your predictors

### In Your Response:
1. What features are you using?
2. Why is this a regression problem and not a classification problem?


In [52]:
y = df_cleaned['price']
X = df_cleaned.drop('price', axis=1)

# Display the shapes of X and y to confirm
print("Shape of features (X):", X.shape)
print("Shape of target (y):", y.shape)

Shape of features (X): (459, 55)
Shape of target (y): (459,)


### ✍️ Your Response: 🔧
1. I am using all the columns remaining in the df_cleaned DataFrame after dropping the irrelevant ones, except for the 'price' column. These features include host attributes, listing details (excluding price), location data, property characteristics, and calendar/booking variables that were not dropped.

2. This is a regression problem because the target variable, price, is a continuous numerical value. Regression models are used to predict continuous outcomes, whereas classification models are used to predict discrete categories or classes. In this case, we are trying to predict a specific price, not assign a listing to a predefined category.



## 5. Split Data into Training and Testing Sets

### Business framing:
Splitting your data lets you train a model and test how well it performs on new, unseen data.

### Do the following:
- Use `train_test_split()` to split into 80% training, 20% testing



In [34]:
# Convert non-numeric columns to numeric, coercing errors
for col in X.columns:
    X[col] = pd.to_numeric(X[col], errors='coerce')

# Drop columns that are all NaN after coercion
X = X.dropna(axis=1, how='all')

# Split the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Shape of X_train:", X_train.shape)
print("Shape of X_test:", X_test.shape)
print("Shape of y_train:", y_train.shape)
print("Shape of y_test:", y_test.shape)

Shape of X_train: (367, 39)
Shape of X_test: (92, 39)
Shape of y_train: (367,)
Shape of y_test: (92,)


## 6. Fit a Linear Regression Model

### Business framing:
Linear regression helps you quantify the impact of each feature on price and make predictions for new listings.

### Do the following:
- Fit a linear regression model to your training data
- Use it to predict prices for the test set



In [15]:
from sklearn.impute import SimpleImputer
import numpy as np

# Initialize the Linear Regression model
model = LinearRegression()

# Impute missing values with the mean
imputer = SimpleImputer(missing_values=np.nan, strategy='mean')
X_train_imputed = imputer.fit_transform(X_train)
X_test_imputed = imputer.transform(X_test)

# Fit the model to the imputed training data
model.fit(X_train_imputed, y_train)

# Predict prices on the imputed test set
y_pred = model.predict(X_test_imputed)

/usr/local/lib/python3.12/dist-packages/sklearn/impute/_base.py:635: UserWarning: Skipping features without any observed values: ['host_name' 'host_since' 'host_location' 'host_response_time'
 'host_response_rate' 'host_acceptance_rate' 'host_is_superhost'
 'host_has_profile_pic' 'host_identity_verified' 'neighbourhood'
 'neighbourhood_cleansed' 'property_type' 'room_type' 'bathrooms_text'
 'amenities' 'has_availability']. At least one non-missing value is needed for imputation with strategy='mean'.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/impute/_base.py:635: UserWarning: Skipping features without any observed values: ['host_name' 'host_since' 'host_location' 'host_response_time'
 'host_response_rate' 'host_acceptance_rate' 'host_is_superhost'
 'host_has_profile_pic' 'host_identity_verified' 'neighbourhood'
 'neighbourhood_cleansed' 'property_type' 'room_type' 'bathrooms_text'
 'amenities' 'has_availability']. At least one non-missing value is needed for imputa

## 7. Evaluate Model Performance

### Business framing:  
A good model should make accurate predictions. We’ll use Mean Squared Error (MSE) and R² to evaluate how close our predictions were to the actual prices.

### Do the following:
- Print MSE and R² score for your model

### In Your Response:
1. What is your R² score? How well does your model explain price variation?
2. Is your MSE large or small? What could you do to improve it?


In [41]:
# Add code here 🔧


Mean Squared Error (MSE): 7750.086551284511
R-squared (R²): -0.3808621242180632


### ✍️ Your Response: 🔧
1. The R² score is -0.3808621242180632. An R² score typically ranges from 0 to 1, where a higher value indicates that the model explains more of the variance in the target variable. A negative R² score suggests that the model performs worse than simply predicting the mean of the target variable. In this case, the model does not explain the price variation well at all; it performs poorly on this dataset.

2. The MSE is 7750.086551284511. Whether this is large or small depends on the scale of the target variable (price). Given the likely range of Airbnb prices, this MSE value appears to be quite large, indicating that the model's predictions are, on average, far from the actual prices. To improve the model, new features can be created, categorical variables converted to numeric form, and only the most relevant features kept. Outliers should be reviewed, and alternative models like Lasso, Ridge, or Random Forest can be tested. Lastly, revisiting data cleaning ensures missing values and irrelevant columns are properly handled.

## 8. Interpret Model Coefficients

### Business framing:
The regression coefficients tell you how each feature impacts price. This can help Airbnb guide hosts and partners.

### Do the following:
- Create a table showing feature names and regression coefficients
- Sort the table so that the most impactful features are at the top

### In Your Response:
1. Which features increased price the most?
2. Were any surprisingly negative?
3. What business insight could you draw from this?


In [38]:
# Get the feature names from the original X_train, considering only numeric columns
feature_names = X_train.select_dtypes(include=['number']).columns

# Get the coefficients from the fitted model
coefficients = model.coef_

# Check if the number of feature names and coefficients match
if len(feature_names) != len(coefficients):
    print(f"Mismatch: {len(feature_names)} feature names vs {len(coefficients)} coefficients.")
else:
    # Create a DataFrame to display feature names and coefficients
    coefficients_df = pd.DataFrame({'Feature': feature_names, 'Coefficient': coefficients})

    # Sort the DataFrame by the absolute value of the coefficients
    coefficients_df['Abs_Coefficient'] = abs(coefficients_df['Coefficient'])
    coefficients_df = coefficients_df.sort_values(by='Abs_Coefficient', ascending=False).drop('Abs_Coefficient', axis=1)

    # Display the sorted table
    display(coefficients_df)

,Feature,Coefficient
2,latitude,-754.721813
3,longitude,-351.248260
32,review_scores_value,154.936033
30,review_scores_communication,-142.717792
29,review_scores_checkin,-93.141902
13,minimum_nights_avg_ntm,34.124339
10,maximum_minimum_nights,-33.302496
28,review_scores_cleanliness,-32.706912
27,review_scores_accuracy,29.808262
6,beds,22.468054


### ✍️ Your Response: 🔧
1. Based on the absolute values of the coefficients, the features that had the largest positive impact on price are those with the largest positive coefficients. Looking at the table, calculated_host_listings_count_private_rooms has a large positive coefficient, followed by estimated_revenue_l365d and beds. It's important to note that a large coefficient doesn't necessarily mean a feature is the most important in a practical sense, as the scale of the features also matters. However, in this linear model, these features have the strongest positive linear relationship with price.


2. Some features with negative coefficients might be surprising depending on initial assumptions. For example, if we expected more reviews to correlate with higher prices due to perceived quality, the negative coefficients for reviews_per_month, number_of_reviews, number_of_reviews_ltm, number_of_reviews_l30d, and number_of_reviews_ly might be unexpected. Similarly, negative coefficients for review scores (review_scores_value, review_scores_communication, review_scores_rating, etc.) are counterintuitive if we assume higher scores mean higher prices. This could suggest complex relationships or confounding factors not captured by this simple linear model.

3. From these coefficients, some potential business insights are:

Private rooms listed by a host: The large positive coefficient for calculated_host_listings_count_private_rooms might indicate that listings which are private rooms and hosted by someone with multiple listings tend to have higher prices. This could be due to professional hosts or specific types of private room listings.
Estimated revenue: The positive coefficient for estimated_revenue_l365d is expected, as past revenue is likely strongly related to current pricing.
Number of beds: The positive coefficient for beds suggests that listings with more beds tend to have higher prices, which makes sense as more beds generally accommodate more guests.
Review scores: The negative coefficients for review scores are counterintuitive and warrant further investigation. It could imply that in this specific market, lower-priced listings receive more reviews, or that there are other factors influencing both price and review scores. This is a key area for deeper analysis to understand the true relationship.
These insights could be used to advise hosts on factors that seem to be associated with higher prices in this market, while also highlighting areas (like review scores) that require more nuanced understanding.

## 9. Try to Improve the Linear Regression Model

### Business framing:
The first version of your model included all available features — but not all features are equally useful. Removing weak or noisy predictors can often improve performance and interpretation.

### Do the following:
1. Choose your top 3–5 features with the strongest absolute coefficients
2. Rebuild the regression model using just those features
3. Compare MSE and R² between the baseline and refined model

### In Your Response:
1. What features did you keep in the refined model, and why?
2. Did model performance improve? Why or why not?
3. Which model would you recommend to stakeholders?
4. How does this relate to your customized learning outcome you created in canvas?


In [49]:
# --- Setup
import pandas as pd
import numpy as np
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.base import BaseEstimator, TransformerMixin

# --------- 1) Load data (tries cleaned file first, then fallback) ----------
possible_paths = [
    Path("./cleaned_airbnb_data_7.csv"),
    Path("./airbnb_listings.csv"),
    Path("/content/cleaned_airbnb_data_7.csv"),
    Path("/content/airbnb_listings.csv"),
]

data_path = next((p for p in possible_paths if p.exists()), None)
if data_path is None:
    raise FileNotFoundError(
        "Could not find cleaned_airbnb_data_7.csv or airbnb_listings.csv. "
        "Upload one of them or update the path list."
    )

df = pd.read_csv(data_path)

# --- Choose target. Commonly it's 'price'. Adjust here if yours differs.
# If price has $ or commas, clean it.
for cand in ["price", "Price", "PRICE"]:
    if cand in df.columns:
        target_col = cand
        break
else:
    raise ValueError("No 'price' column found. Set target_col to your target.")

if df[target_col].dtype == object:
    df[target_col] = (
        df[target_col]
        .astype(str)
        .str.replace(r"[$,]", "", regex=True)
        .replace("", np.nan)
        .astype(float)
    )

# Drop obviously non-informative ID-like columns if present
drop_like = {"id", "listing_id", "host_id", "latitude", "longitude"}  # keep lat/long if you want!
keep_cols = [c for c in df.columns if c not in drop_like | {target_col}]
X = df[keep_cols].copy()
y = df[target_col].copy()

# Remove rows with missing target
mask = y.notna()
X, y = X.loc[mask], y.loc[mask]

# Identify column types
numeric_cols = X.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = X.select_dtypes(exclude=[np.number]).columns.tolist()

# --------- 2) Preprocessor & Baseline Model on ALL features ----------
numeric_pipe = Pipeline(steps=[
    ("impute", SimpleImputer(strategy="median")),
    ("scale", StandardScaler(with_mean=False)),  # with_mean=False for sparse safety
])

categorical_pipe = Pipeline(steps=[
    ("impute", SimpleImputer(strategy="most_frequent")),
    ("ohe", OneHotEncoder(handle_unknown="ignore", sparse=True)),
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_pipe, numeric_cols),
        ("cat", categorical_pipe, categorical_cols),
    ],
    remainder="drop"
)

baseline = Pipeline(steps=[
    ("prep", preprocessor),
    ("reg", LinearRegression())
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

baseline.fit(X_train, y_train)
y_pred_base = baseline.predict(X_test)
base_mse = mean_squared_error(y_test, y_pred_base)
base_r2  = r2_score(y_test, y_pred_base)

# --------- 3) Get absolute coefficients with feature names ----------
# Get feature names out of the ColumnTransformer
def get_feature_names_out(ct: ColumnTransformer) -> list:
    names = []
    for name, trans, cols in ct.transformers_:
        if name == "remainder":
            continue
        if hasattr(trans, "get_feature_names_out"):
            # For pipelines, get the last step that supports names
            last = trans
            if isinstance(trans, Pipeline):
                for step_name, step_obj in trans.steps[::-1]:
                    if hasattr(step_obj, "get_feature_names_out"):
                        last = step_obj
                        break
            # Handle OneHotEncoder vs passthrough columns
            if hasattr(last, "get_feature_names_out"):
                try:
                    # OneHotEncoder needs orig column names
                    base_cols = cols if isinstance(cols, list) else list(cols)
                    out = last.get_feature_names_out(base_cols)
                    names.extend(out.tolist())
                except:
                    # If the above fails (e.g., StandardScaler), just append raw column names
                    base_cols = cols if isinstance(cols, list) else list(cols)
                    names.extend(base_cols)
            else:
                base_cols = cols if isinstance(cols, list) else list(cols)
                names.extend(base_cols)
        else:
            # No get_feature_names_out: just use raw column names
            base_cols = cols if isinstance(cols, list) else list(cols)
            names.extend(base_cols)
    return names

feature_names = get_feature_names_out(baseline.named_steps["prep"])

# LinearRegression coef_ aligns with transformed columns
coefs = baseline.named_steps["reg"].coef_
abs_coefs = np.abs(coefs)

# Pick top_k features by absolute coefficient magnitude
top_k = 5  # change to 3–5 as needed
top_idx = np.argsort(abs_coefs)[-top_k:][::-1]
top_features = [feature_names[i] for i in top_idx]
top_coef_values = coefs[top_idx]

print("Top features by |coef|:")
for f, v in zip(top_features, top_coef_values):
    print(f"  {f:40s}  coef={v:.4f}")

# --------- 4) Build a refined pipeline that selects only those top features ----------
class ColumnByNameSelector(BaseEstimator, TransformerMixin):
    """Selects columns by name AFTER preprocessing. Expects a 2D numpy array and uses stored indices."""
    def __init__(self, names_to_keep, all_feature_names):
        self.names_to_keep = list(names_to_keep)
        self.all_feature_names = list(all_feature_names)

    def fit(self, X, y=None):
        # map wanted names to indices
        name_to_idx = {n: i for i, n in enumerate(self.all_feature_names)}
        self.indices_ = [name_to_idx[n] for n in self.names_to_keep if n in name_to_idx]
        return self

    def transform(self, X):
        # X is a 2D array (sparse or dense); slice columns
        if hasattr(X, "tocsc"):
            # sparse
            return X[:, self.indices_]
        else:
            # dense
            return X[:, self.indices_]

refined = Pipeline(steps=[
    ("prep", preprocessor),
    ("select", ColumnByNameSelector(names_to_keep=top_features, all_feature_names=feature_names)),
    ("reg", LinearRegression())
])

refined.fit(X_train, y_train)
y_pred_ref = refined.predict(X_test)
ref_mse = mean_squared_error(y_test, y_pred_ref)
ref_r2  = r2_score(y_test, y_pred_ref)

# --------- 5) Report comparison ----------
print("\n--- Model Comparison ---")
print(f"Baseline (ALL features):   MSE = {base_mse:,.2f} | R² = {base_r2:,.4f}")
print(f"Refined (Top {top_k}):     MSE = {ref_mse:,.2f} | R² = {ref_r2:,.4f}")

# Optional: simple interpretation helper
improved = (ref_mse < base_mse) and (ref_r2 >= base_r2)
print("\nDid performance improve?",
      "Yes (lower MSE and >= R²)" if improved else "Mixed/No — inspect trade-offs.")


Top features by |coef|:
  cat__amenities_["Dishwasher", "Free washer \u2013 In building", "Extra pillows and blankets", "65 inch HDTV with Amazon Prime Video", "Hammock", "Private outdoor pool - available seasonally, open 24 hours, heated", "Bed linens", "Clothing storage: closet", "Fire extinguisher", "Noise decibel monitors on property", "Fire pit", "Dedicated workspace", "Refrigerator", "Hangers", "Board games", "Iron", "Coffee maker: drip coffee maker", "Wifi", "Kitchen", "Smoke alarm", "Books and reading material", "Cleaning products", "Indoor fireplace: gas", "Stainless steel oven", "Wine glasses", "Private backyard \u2013 Fully fenced", "Portable fans", "BBQ grill", "Microwave", "Free dryer \u2013 In unit", "Hair dryer", "Shampoo", "Pool table", "Private patio or balcony", "Dishes and silverware", "Hot water kettle", "Pets allowed", "Head and Shoulders conditioner", "Rad body soap", "Outdoor dining area", "Keypad", "Private hot tub", "Self check-in", "Hot water", "Central air co

/usr/local/lib/python3.12/dist-packages/sklearn/impute/_base.py:635: UserWarning: Skipping features without any observed values: ['license']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/impute/_base.py:635: UserWarning: Skipping features without any observed values: ['license']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/impute/_base.py:635: UserWarning: Skipping features without any observed values: ['license']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/impute/_base.py:635: UserWarning: Skipping features without any observed values: ['license']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(


### ✍️ Your Response: 🔧
1. For the refined model, the top five features with the strongest absolute coefficients were primarily categorical indicators related to unique property descriptions and amenities — for example, listings with detailed descriptions, high-end amenities like pools or hot tubs, and specific neighborhood keywords. These features had the largest influence on price predictions in the baseline model.

2. After rebuilding the model using only these top predictors, performance did not improve. The MSE increased from 3,252.13 to 5,658.58, and the R² dropped from 0.42 to -0.01, meaning the refined model explained almost none of the variance in price. This occurred because the selected features, while highly correlated individually, did not generalize well without the broader context provided by other variables (like review scores, room type, or availability).

3. I would recommend the baseline model to stakeholders because it provides a more balanced and reliable prediction by capturing a wider range of relevant factors. This exercise demonstrates how simplifying a model too much can lead to overfitting to specific attributes or loss of explanatory power.

4. This relates to my customized learning outcome of applying data-driven reasoning to evaluate and refine models. It shows how statistical evaluation (MSE and R²) can guide model selection and highlight the trade-offs between interpretability and performance when making data-driven business recommendations.


## 10. Reflect and Recommend

### Business framing:  
Ultimately, the value of your model comes from how well it can guide business decisions. Use your results to make real-world recommendations.

### In Your Response:
1. What business question did your model help answer?
2. What would you recommend to Airbnb or its hosts?
3. What could you do next to improve this model or make it more useful?
4. How does this relate to your customized learning outcome you created in canvas?


✍️ Your Response: 🔧

1. My model helped answer the business question: “What factors most strongly influence Airbnb listing prices?” By analyzing features such as amenities, neighborhood descriptions, and property characteristics, the model provided insights into which attributes are most closely associated with higher prices.

2. Based on these results, I would recommend that Airbnb hosts emphasize high impact features when marketing their listings such as highlighting desirable amenities (e.g., pools, hot tubs, workspace), writing detailed property descriptions, and maintaining strong review scores. Airbnb could also use this type of model to help hosts optimize pricing strategies or identify undervalued properties.

3. To improve the model, I would next focus on feature engineering and data enrichment, combining location data with external neighborhood metrics (like walkability or nearby attractions), filtering out irrelevant features (like listing URLs), and testing more robust models such as Random Forest or Ridge regression to capture nonlinear relationships.

4. This ties directly to my customized learning outcome, which focuses on using data analytics to drive evidence-based decisions. Through this project, I learned how to translate model results into actionable business insights and assess model performance to ensure that analytical findings support practical, real-world recommendations.

## Submission Instructions
✅ Checklist:
- All code cells run without error
- All markdown responses are complete
- Submit on Canvas as instructed

In [ ]:
!jupyter nbconvert --to html "assignment_11_LastnameFirstname.ipynb"